# Data Extraction for all Conflicts

**Output:** `data/raw/{conflict}.csv` · `data/merged/all_conflicts.csv`

> **Note:** The dataset used in this project is generated programmatically by this notebook through a series of SPARQL queries to the [Wikidata Query Service](https://query.wikidata.org/). No external data files are required as input. Running the notebook from top to bottom will fetch, process, and save all conflict-level weapon and manufacturer data.


In [1]:
import pandas as pd
import requests
import time
import os
import json
from pathlib import Path

# Output folders
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/merged').mkdir(parents=True, exist_ok=True)


SPARQL_URL = 'https://query.wikidata.org/sparql'
HEADERS = {
    'User-Agent': 'KEN4256-DataBot/1.0 (student research project)',
    'Accept':     'application/sparql-results+json'
}

# Conflicts: name → QID Wikidata
CONFLICTS = {
    'ukraine':          'Q110999040',   
    'gaza':             'Q122962941',  
    'syria':            'Q178810',      
    'lebanon_2006':     'Q49104',       
    'nagorno_karabakh': 'Q99659280',    
    'iraq':             'Q545449',      
    'kosovo_1999': 'Q190029',
    'afghanistan': 'Q182865',
    'iran_iraq': 'Q82664',


}

print(f'Conflicts to extract: {len(CONFLICTS)}')
for name, qid in CONFLICTS.items():
    print(f'  {name:<20} wd:{qid}')


Conflicts to extract: 9
  ukraine              wd:Q110999040
  gaza                 wd:Q122962941
  syria                wd:Q178810
  lebanon_2006         wd:Q49104
  nagorno_karabakh     wd:Q99659280
  iraq                 wd:Q545449
  kosovo_1999          wd:Q190029
  afghanistan          wd:Q182865
  iran_iraq            wd:Q82664


In [2]:
# Query SPARQL 

COLUMNS = [
    'conflict', 'conflict_qid',
    'participant', 'participantLabel',
    'weapon', 'weaponLabel',
    'type', 'typeLabel',
    'manufacturer', 'manufacturerLabel',
    'manufacturerCountry', 'manufacturerCountryLabel',
    'origin', 'originLabel',
]

QUERY_TEMPLATE = '''
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX bd:  <http://www.bigdata.com/rdf#>
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT DISTINCT
  ?participant ?participantLabel
  ?weapon ?weaponLabel
  ?type ?typeLabel
  ?manufacturer ?manufacturerLabel
  ?manufacturerCountry ?manufacturerCountryLabel
  ?origin ?originLabel
WHERE {{
  BIND(wd:{conflict_qid} AS ?conflict)

  ?conflict wdt:P710 ?participant .
  ?weapon   wdt:P137 ?participant .

  {{ ?weapon wdt:P176 ?manufacturer . }}
  UNION
  {{ ?weapon wdt:P495 ?origin . }}

  OPTIONAL {{
    ?weapon wdt:P31 ?type .
    VALUES ?type {{
      wd:Q15142894 wd:Q18487055 wd:Q124056273
      wd:Q15056995 wd:Q15056993
      wd:Q100710213 wd:Q100709275
      wd:Q178550   wd:Q465264
      wd:Q913      wd:Q1330890
      wd:Q12876    wd:Q12796
      wd:Q559026   wd:Q1133012
      wd:Q484000   wd:Q2735392
      wd:Q137188246 wd:Q45296117
      wd:Q22704163  wd:Q42314054
    }}
  }}

  OPTIONAL {{
    ?weapon wdt:P176 ?manufacturer .
    OPTIONAL {{ ?manufacturer wdt:P17 ?manufacturerCountry . }}
  }}
  OPTIONAL {{ ?weapon wdt:P495 ?origin . }}

  MINUS {{ VALUES ?weapon {{ wd:Q906937 }} }}

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,it". }}
}}
LIMIT 5000
'''

def run_sparql(query: str, retries: int = 3, backoff: float = 5.0) -> list:
    for attempt in range(retries):
        try:
            r = requests.get(
                SPARQL_URL,
                params={'query': query},
                headers=HEADERS,
                timeout=120
            )
            if r.status_code == 200:
                return r.json()['results']['bindings']
            elif r.status_code == 429:
                wait = backoff * (2 ** attempt)
                print(f'    Rate limited (429). Attendo {wait:.0f}s...')
                time.sleep(wait)
            else:
                print(f'    HTTP {r.status_code} — retry {attempt+1}/{retries}')
                time.sleep(backoff)
        except requests.exceptions.Timeout:
            print(f'    Timeout — retry {attempt+1}/{retries}')
            time.sleep(backoff)
        except Exception as e:
            print(f'    Errore: {e} — retry {attempt+1}/{retries}')
            time.sleep(backoff)
    raise RuntimeError(f'Query fallita dopo {retries} tentativi')


def bindings_to_df(bindings: list, conflict_name: str, conflict_qid: str) -> pd.DataFrame:
    if not bindings:
        return pd.DataFrame(columns=COLUMNS)
    rows = []
    for b in bindings:
        rows.append({
            'conflict':                 conflict_name,
            'conflict_qid':             f'http://www.wikidata.org/entity/{conflict_qid}',
            'participant':              b.get('participant',            {}).get('value', ''),
            'participantLabel':         b.get('participantLabel',       {}).get('value', ''),
            'weapon':                   b.get('weapon',                 {}).get('value', ''),
            'weaponLabel':              b.get('weaponLabel',            {}).get('value', ''),
            'type':                     b.get('type',                   {}).get('value', ''),
            'typeLabel':                b.get('typeLabel',              {}).get('value', ''),
            'manufacturer':             b.get('manufacturer',           {}).get('value', ''),
            'manufacturerLabel':        b.get('manufacturerLabel',      {}).get('value', ''),
            'manufacturerCountry':      b.get('manufacturerCountry',    {}).get('value', ''),
            'manufacturerCountryLabel': b.get('manufacturerCountryLabel',{}).get('value',''),
            'origin':                   b.get('origin',                 {}).get('value', ''),
            'originLabel':              b.get('originLabel',            {}).get('value', ''),
        })
    return pd.DataFrame(rows)


print('Query template e helpers pronti.')


Query template e helpers pronti.


In [3]:
# Download for each conflict

DELAY_BETWEEN_CONFLICTS = 10
results_summary = []

for conflict_name, conflict_qid in CONFLICTS.items():

    out_path = Path(f'data/raw/{conflict_name}.csv')

    if out_path.exists():
        try:
            existing = pd.read_csv(out_path)
            if len(existing) > 0:
                n_w = existing['weapon'].nunique() if 'weapon' in existing.columns else 0
                n_m = existing['manufacturer'].replace('', pd.NA).dropna().nunique() if 'manufacturer' in existing.columns else 0
                print(f'[SKIP]  {conflict_name:<20} gia presente — {len(existing)} righe | {n_w} weapon | {n_m} manufacturer')
                results_summary.append({'conflict': conflict_name, 'qid': conflict_qid,
                    'rows': len(existing), 'weapons': n_w, 'manufacturers': n_m, 'status': 'cached'})
                continue
            else:
                print(f'[EMPTY] {conflict_name:<20} CSV vuoto dalla run precedente — riscarico')
                out_path.unlink()
        except pd.errors.EmptyDataError:
            print(f'[EMPTY] {conflict_name:<20} CSV corrotto — riscarico')
            out_path.unlink()

    print(f'\n[DOWNLOAD] {conflict_name} (wd:{conflict_qid})')

    try:
        query    = QUERY_TEMPLATE.format(conflict_qid=conflict_qid)
        bindings = run_sparql(query)
        df       = bindings_to_df(bindings, conflict_name, conflict_qid)

        if len(df) == 0:
            print(f'  WARNING: 0 risultati per wd:{conflict_qid}')
            print(f'  Cause possibili:')
            print(f'    1. QID errato')
            print(f'    2. Conflitto senza P710 (participant) su Wikidata')
            print(f'    3. Partecipanti senza asset linkati con P137')
            results_summary.append({'conflict': conflict_name, 'qid': conflict_qid,
                'rows': 0, 'weapons': 0, 'manufacturers': 0,
                'status': 'WARNING: 0 results'})
            # non salvare CSV vuoto — cosi' al prossimo run ritenta
        else:
            df.to_csv(out_path, index=False)
            n_w = df['weapon'].nunique()
            n_m = df['manufacturer'].replace('', pd.NA).dropna().nunique()
            print(f'  {len(df)} righe | {n_w} weapon unici | {n_m} manufacturer unici')
            print(f'  Salvato: {out_path}')
            results_summary.append({'conflict': conflict_name, 'qid': conflict_qid,
                'rows': len(df), 'weapons': n_w, 'manufacturers': n_m,
                'status': 'downloaded'})

    except Exception as e:
        print(f'  ERRORE: {e}')
        results_summary.append({'conflict': conflict_name, 'qid': conflict_qid,
            'rows': 0, 'weapons': 0, 'manufacturers': 0, 'status': f'error: {e}'})

    if conflict_name != list(CONFLICTS.keys())[-1]:
        print(f'  Attendo {DELAY_BETWEEN_CONFLICTS}s...')
        time.sleep(DELAY_BETWEEN_CONFLICTS)


print('\n' + '='*60)
print('RECAP')
print('='*60)
summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('data/download_summary.csv', index=False)

problemi = summary_df[summary_df['rows'] == 0]
if len(problemi) > 0:
    print(f'\n{len(problemi)} conflicts with 0 results:')
    for _, r in problemi.iterrows():
        print(f'  {r["conflict"]} (wd:{r["qid"]})')



[DOWNLOAD] ukraine (wd:Q110999040)
  142 righe | 89 weapon unici | 75 manufacturer unici
  Salvato: data/raw/ukraine.csv
  Attendo 10s...

[DOWNLOAD] gaza (wd:Q122962941)
  79 righe | 60 weapon unici | 30 manufacturer unici
  Salvato: data/raw/gaza.csv
  Attendo 10s...

[DOWNLOAD] syria (wd:Q178810)
  48 righe | 34 weapon unici | 27 manufacturer unici
  Salvato: data/raw/syria.csv
  Attendo 10s...

[DOWNLOAD] lebanon_2006 (wd:Q49104)
  70 righe | 57 weapon unici | 28 manufacturer unici
  Salvato: data/raw/lebanon_2006.csv
  Attendo 10s...

[DOWNLOAD] nagorno_karabakh (wd:Q99659280)
  13 righe | 7 weapon unici | 9 manufacturer unici
  Salvato: data/raw/nagorno_karabakh.csv
  Attendo 10s...

[DOWNLOAD] iraq (wd:Q545449)
  314 righe | 130 weapon unici | 116 manufacturer unici
  Salvato: data/raw/iraq.csv
  Attendo 10s...

[DOWNLOAD] kosovo_1999 (wd:Q190029)
  197 righe | 80 weapon unici | 93 manufacturer unici
  Salvato: data/raw/kosovo_1999.csv
  Attendo 10s...

[DOWNLOAD] afghanistan (

In [4]:
# Merge all CSV files into a single dataset

all_dfs = []
for conflict_name in CONFLICTS.keys():
    path = Path(f'data/raw/{conflict_name}.csv')
    if path.exists():
        df = pd.read_csv(path)
        all_dfs.append(df)
    else:
        print(f'  ATTENZIONE: {path} non trovato — saltato dal merge')

merged = pd.concat(all_dfs, ignore_index=True)


before = len(merged)
merged_dedup = merged.drop_duplicates(
    subset=['conflict', 'weapon', 'manufacturer', 'origin']
)
after = len(merged_dedup)

merged_dedup.to_csv('data/merged/all_conflicts.csv', index=False)

print(f'Righe totali pre-dedup:  {before}')
print(f'Righe totali post-dedup: {after}')
print(f'Duplicate rimosse:       {before - after}')
print(f'\nSalvato: data/merged/all_conflicts.csv')
print(f'\nDistribuzione righe per conflitto:')
print(merged_dedup.groupby('conflict')['weapon'].nunique()
      .sort_values(ascending=False)
      .rename('weapon_unici')
      .reset_index()
      .to_string(index=False))


Righe totali pre-dedup:  1013
Righe totali post-dedup: 835
Duplicate rimosse:       178

Salvato: data/merged/all_conflicts.csv

Distribuzione righe per conflitto:
        conflict  weapon_unici
            iraq           130
         ukraine            89
     kosovo_1999            80
     afghanistan            77
            gaza            60
    lebanon_2006            57
           syria            34
nagorno_karabakh             7
       iran_iraq             5


In [5]:
# Quality check del dataset merged 

df = pd.read_csv('data/merged/all_conflicts.csv')

print('=' * 60)
print('QUALITY CHECK — all_conflicts.csv')
print('=' * 60)

print(f'\nDimension: {df.shape[0]} rows × {df.shape[1]} columns')

# coverage per conflict
print('\n── Coverage per conflict:')
cov = df.groupby('conflict').agg(
    righe              = ('weapon', 'count'),
    weapon_unici       = ('weapon', 'nunique'),
    manufacturer_unici = ('manufacturer', lambda x: x.replace('', pd.NA).dropna().nunique()),
    pct_con_manu       = ('manufacturer', lambda x: f"{(x.replace('',pd.NA).notna().mean()*100):.1f}%"),
    pct_con_type       = ('type',         lambda x: f"{(x.replace('',pd.NA).notna().mean()*100):.1f}%"),
).reset_index()
print(cov.to_string(index=False))

# asset without manufacturer and origin
no_manu_no_origin = df[
    (df['manufacturer'].replace('', pd.NA).isna()) &
    (df['origin'].replace('', pd.NA).isna())
]
print(f'\n── Asset without manufacturer and origin: {len(no_manu_no_origin)}')
if len(no_manu_no_origin) > 0:
    print('  (these will be excluded from the ownership pipeline — Wikidata gap)')

# weapon uniques without type (type=NULL)
no_type = df[df['type'].replace('', pd.NA).isna()]['weapon'].nunique()
print(f'\n── Weapon uniques without type (type=NULL): {no_type}')
print('  (included in the dataset, type classified as fallback in M3)')

# top 10 manufacturer by number of conflicts covered
print('\n── Top 10 manufacturer by number of conflicts covered:')
top_manu = (
    df[df['manufacturer'].replace('', pd.NA).notna()]
    .groupby(['manufacturerLabel', 'manufacturer'])['conflict']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'conflict': 'n_conflitti'})
)
print(top_manu[['manufacturerLabel', 'n_conflitti']].to_string(index=False))

# top 10 origin country cross-conflict
print('\n── Top 10 origin countries by frequency:')
top_origin = (
    df[df['originLabel'].replace('', pd.NA).notna()]
    .groupby('originLabel')['weapon']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)
print(top_origin.to_string())


QUALITY CHECK — all_conflicts.csv

Dimension: 835 rows × 14 columns

── Coverage per conflict:
        conflict  righe  weapon_unici  manufacturer_unici pct_con_manu pct_con_type
     afghanistan    117            77                  59        92.3%        82.9%
            gaza     73            60                  30        75.3%        60.3%
       iran_iraq     13             5                   8        92.3%       100.0%
            iraq    217           130                 116        94.9%        86.2%
     kosovo_1999    149            80                  93        92.6%        83.2%
    lebanon_2006     70            57                  28        75.7%        60.0%
nagorno_karabakh     12             7                   9        83.3%        91.7%
           syria     48            34                  27        83.3%        66.7%
         ukraine    136            89                  75        89.7%        89.7%

── Asset without manufacturer and origin: 0

── Weapon uniques w

In [6]:
# Prepare clean dataset for ownership reconstruction
# Extracts the unique manufacturer cross-conflict to pass to the second notebook

df = pd.read_csv('data/merged/all_conflicts.csv')

import re
qid_like = re.compile(r'^Q\d+$')
def is_placeholder(x):
    if pd.isna(x) or str(x).strip() == '': return True
    return bool(qid_like.match(str(x).strip()))

manu_df = (
    df[df['manufacturer'].replace('', pd.NA).notna()]
    .copy()
)
manu_df = manu_df[~manu_df['manufacturerLabel'].apply(is_placeholder)]

manufacturers_all = (
    manu_df
    .groupby(['manufacturer', 'manufacturerLabel', 'manufacturerCountry', 'manufacturerCountryLabel'])
    .agg(conflicts=('conflict', lambda x: '|'.join(sorted(x.unique()))))
    .reset_index()
    .sort_values('manufacturerLabel')
)

manufacturers_all.to_csv('data/merged/manufacturers_all_conflicts.csv', index=False)

print(f'Manufacturer unici cross-conflitto: {len(manufacturers_all)}')
print(f'Salvato: data/merged/manufacturers_all_conflicts.csv')
print(f'\nManufacturer per numero di conflitti coperti:')
print(
    manufacturers_all['conflicts']
    .apply(lambda x: len(x.split('|')))
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis('n_conflitti')
    .reset_index(name='n_manufacturer')
    .to_string(index=False)
)


Manufacturer unici cross-conflitto: 196
Salvato: data/merged/manufacturers_all_conflicts.csv

Manufacturer per numero di conflitti coperti:
 n_conflitti  n_manufacturer
           6               5
           5               6
           4              25
           3              15
           2              72
           1              73
